In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from torch.utils.data import TensorDataset, DataLoader

In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [3]:
fashion_mnist = fetch_openml(name="Fashion-MNIST", as_frame=False)

X = torch.FloatTensor(fashion_mnist.data / 255.0)
y = torch.LongTensor(fashion_mnist.target.astype(int))

In [4]:
X.shape, y.shape

(torch.Size([70000, 784]), torch.Size([70000]))

In [5]:
# Train -> 55,000 | Validation -> 5,000 | Test ->10,000
train_set = TensorDataset(X[:55_000], y[:55_000])
valid_set = TensorDataset(X[55_000:60_000], y[55_000:60_000])
test_set  = TensorDataset(X[60_000:], y[60_000:])

BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE)

In [6]:
len(train_set), len(valid_set), len(test_set)

(55000, 5000, 10000)

In [7]:
def use_he_init(module):
  if isinstance(module, nn.Linear):
    nn.init.kaiming_uniform_(module.weight)
    nn.init.zeros_(module.bias)

model = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
model.apply(use_he_init)

Sequential(
  (0): Linear(in_features=784, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=128, bias=True)
  (3): ReLU()
  (4): Linear(in_features=128, out_features=64, bias=True)
  (5): ReLU()
  (6): Linear(in_features=64, out_features=10, bias=True)
)

In [8]:
model = nn.Sequential(nn.Linear(784, 256), nn.LeakyReLU(negative_slope=0.2))
model.apply(use_he_init)

Sequential(
  (0): Linear(in_features=784, out_features=256, bias=True)
  (1): LeakyReLU(negative_slope=0.2)
)

In [9]:
torch.manual_seed(42)
alpha = 0.2
model = nn.Sequential(nn.Linear(784, 256), nn.LeakyReLU(negative_slope=alpha))
nn.init.kaiming_uniform_(model[0].weight, alpha, nonlinearity="leaky_relu")
model(torch.rand(2, 784)).shape

torch.Size([2, 256])

In [10]:
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(784, 256), nn.ELU())
nn.init.kaiming_uniform_(model[0].weight)
model(torch.rand(2, 784)).shape

torch.Size([2, 256])

In [11]:
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(784, 256), nn.SELU())
nn.init.kaiming_uniform_(model[0].weight)
model(torch.rand(2, 784)).shape

torch.Size([2, 256])

# Activation Functions

In [12]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def swish(z, beta=1):
    return z * sigmoid(beta * z)

def approx_gelu(z):
    return swish(z, beta=1.702)

def softplus(z):
    return np.log(1 + np.exp(z))

def mish(z):
    return z * np.tanh(softplus(z))

def relu_squared(z):
    return np.maximum(0, z)**2

# SwiGLU

In [13]:
class SwiGLU(nn.Module):
    def __init__(self, beta=1.0):
        super().__init__()
        self.beta = beta

    def forward(self, x):
        z1, z2 = x.chunk(2, dim=-1)
        param_swish = z1 * torch.sigmoid(self.beta * z1)
        return param_swish * z2

torch.manual_seed(42)
model = nn.Sequential(nn.Linear(784, 2 * 256), SwiGLU(beta=0.2))
nn.init.kaiming_uniform_(model[0].weight)
model(torch.rand(2, 784)).shape

torch.Size([2, 256])

# ReLU2

In [14]:
import torch.nn.functional as F

class ReLU2(nn.Module):
    def forward(self, x):
        return F.relu(x).square()

torch.manual_seed(42)
model = nn.Sequential(nn.Linear(784, 256), ReLU2())
nn.init.kaiming_uniform_(model[0].weight)
model(torch.rand(2, 784)).shape

torch.Size([2, 256])

# Batch Normalization

In [28]:
from torch.nn.modules.batchnorm import BatchNorm1d
torch.manual_seed(42)

model = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(784),
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.BatchNorm1d(256),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.BatchNorm1d(128),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.BatchNorm1d(64),
    nn.Linear(64, 10)
)

In [29]:
dict(model[1].named_parameters()).keys()

dict_keys(['weight', 'bias'])

In [30]:
dict(model[1].named_buffers()).keys()

dict_keys(['running_mean', 'running_var', 'num_batches_tracked'])

In [31]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256, bias=False),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Linear(256, 128, bias=False),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Linear(128, 64, bias=False),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

# Layer Normalization

In [32]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256),
    nn.LayerNorm(256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.LayerNorm(128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.LayerNorm(64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

In [37]:
def train_model(model, criterion, optimizer, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)   # y artıq LongTensor-dur
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch+1}/{n_epochs} | Loss: {total_loss:.4f}")

In [38]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds = model(X_batch).argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total   += y_batch.size(0)
    return correct / total

In [39]:
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [40]:
model.to(device)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (3): ReLU()
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (6): ReLU()
  (7): Linear(in_features=128, out_features=64, bias=True)
  (8): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (9): ReLU()
  (10): Linear(in_features=64, out_features=10, bias=True)
)

In [41]:
train_model(model, xentropy, optimizer, train_loader, 10)

  Epoch 1/10 | Loss: 273.3145
  Epoch 2/10 | Loss: 264.9031
  Epoch 3/10 | Loss: 256.5509
  Epoch 4/10 | Loss: 245.7157
  Epoch 5/10 | Loss: 244.2090
  Epoch 6/10 | Loss: 234.9724
  Epoch 7/10 | Loss: 228.7145
  Epoch 8/10 | Loss: 221.9225
  Epoch 9/10 | Loss: 215.5646
  Epoch 10/10 | Loss: 210.1368


In [42]:
val_acc  = evaluate(model, valid_loader)
val_acc

0.8874

In [43]:
test_acc = evaluate(model, test_loader)
test_acc

0.8834